# Milestone 1: data, fixed split, zero-shot baseline
Run on **Kaggle → GPU T4 x1, Internet ON**. Runtime ~25-35 min (mostly two model downloads + validation generation).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone YOUR GitHub repo (change the URL if your repo name differs)
!git clone https://github.com/UsmanBari/trial-eligibility-transformer.git
%cd trial-eligibility-transformer
!pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
# Unit tests (no GPU needed): split rule, leakage, parsing, metrics, truncation
!python -m pytest -q tests

## 1. Data preparation + split manifest

In [ ]:
!python scripts/prepare_data.py

## 2. Zero-shot baseline on VALIDATION (test split stays untouched)

In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen2.5-1.5B-Instruct --split val --tag baseline_qwen25_1p5b

In [ ]:
!python scripts/evaluate.py --model HuggingFaceTB/SmolLM2-1.7B-Instruct --split val --tag baseline_smollm2_1p7b

## 3. Comparison table

In [ ]:
import json
for tag in ['baseline_qwen25_1p5b', 'baseline_smollm2_1p7b']:
    m = json.load(open(f'results/{tag}/metrics.json'))
    ri = json.load(open(f'results/{tag}/run_info.json'))
    o = m['overall']
    print(f"{tag:28s} acc={o['classification']['accuracy']:.3f} macroF1={o['classification']['macro_f1']:.3f} "
          f"invalid={o['classification']['n_invalid']:3d} evMicroF1={o['evidence']['micro']['f1']:.3f} "
          f"params={ri['model']['n_params']/1e9:.2f}B peakMem={ri['peak_gpu_mem_mb']:.0f}MB time={ri['wall_time_s']:.0f}s")

In [ ]:
# A few raw outputs, to see how the base model behaves
!head -n 3 results/baseline_qwen25_1p5b/predictions.jsonl

## 4. Download results
Download `m1_results.zip` from the Output panel (right side), then unzip it into your local repo and commit, or just send the two `metrics.json` + the manifest to Claude.

In [ ]:
!zip -qr /kaggle/working/m1_results.zip data/splits results
!ls -la /kaggle/working